# 05 — Valuation and Report

This notebook covers the final analytical and output stages:

1. **DCF valuation** — projected revenue × FCF margin, discounted cash flows
2. **Scenario analysis** — bear / base / bull with probability weights
3. **Sensitivity table** — WACC × terminal growth grid
4. **Reverse-DCF grid** — what growth + margin the market is pricing in
5. **Implied solvers** — one-variable implied CAGR and implied margin
6. **Recommendation eligibility gate** — formal checklist before any rating is issued
7. **Recommendation** — scorecard-based Buy / Hold / Sell (or Not Rated)
8. **Charts** — 6–8 decision-useful exhibits
9. **Report generation** — Markdown + PDF/HTML output (3 report modes)
10. **Audit consistency check** — cross-file contradiction detection

Modules: `src/valuation.py`, `src/charts.py`, `src/report_utils.py`, `src/audit_utils.py`

---

### ⚠️ Recommendation Eligibility Gate

Before any Buy/Hold/Sell rating is issued, the pipeline evaluates a **recommendation
eligibility gate** that checks all blocking requirements:

- `DataQualityStatus` must be PASS or PASS_WITH_WARNINGS
- Validated market price must be available
- DCF assumptions must be documented and internally consistent
- No unresolved share-count or split-basis issues
- No report-date/lookahead violations
- No audit contradictions between output files

**Non-blocking components** (ML signal, NLP signal, peer multiples, segment charts)
can have status `diagnostic_only` without blocking a formal rating. They are excluded
from score direction and disclosed, but the report can still carry a formal rating.

The gate produces a `RecommendationStatus` → `ReportMode`:
- `formal_rating` → Buy / Hold / Sell with full scorecard
- `diagnostic_not_rated` → "Not Rated — Data Validation Required" with diagnostic outputs
- `failed` → pipeline error, report generation failed

After report generation, a **post-report audit consistency check** cross-validates
all output files for contradictions. Results are written to `outputs/audit_status.json`.

> **Note:** The authoritative reproducibility path is `python scripts/run_pipeline.py --ticker NVDA --report-date 2026-05-01 --price-date 2026-05-01 --output-format both`. This notebook is an explanatory wrapper that inspects the same pipeline outputs. It does not generate final artifacts independently.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
from pathlib import Path

from src.config import get_default_config, ScenarioAssumptions
from src.valuation import ValuationModule
from src.charts import ChartGenerator
from src.report_utils import ReportGenerator
from src.audit_utils import AuditModule

In [ ]:
config = get_default_config()
val = ValuationModule(config)

## 1. DCF Valuation

`compute_dcf()` projects revenue using a CAGR, applies a linearly
interpolated FCF margin (start → terminal), discounts at WACC, and
adds a Gordon Growth terminal value.

**FCF = projected_revenue × fcf_margin**

This is intentionally transparent — every assumption is explicit.

In [ ]:
# Use the base-case scenario assumptions
base_assumptions = config.scenarios["base"]

print("Base-case assumptions:")
print(f"  Revenue CAGR:       {base_assumptions.revenue_cagr:.0%}")
print(f"  FCF margin start:   {base_assumptions.fcf_margin_start:.0%}")
print(f"  FCF margin terminal:{base_assumptions.fcf_margin_terminal:.0%}")
print(f"  Terminal growth:    {base_assumptions.terminal_growth:.1%}")
print(f"  SBC treatment:     {base_assumptions.sbc_treatment}")
print(f"  Probability:       {base_assumptions.probability:.0%}")

In [ ]:
# Load actual values from pipeline outputs
# The authoritative run path is: python scripts/run_pipeline.py
# This notebook is explanatory — it inspects the same pipeline state.
import pandas as pd
metrics = pd.read_csv('data/processed/nvda_metrics.csv')
_rev = metrics[(metrics['metric_name']=='revenue') & (metrics['fiscal_period'].str.match(r'^FY\d{4}$'))]
base_revenue = float(_rev.sort_values('fiscal_period').iloc[-1]['metric_value'])  # latest annual revenue
wacc = config.wacc

# Extract net cash from validated XBRL data (cash - debt)
_cash = metrics[(metrics['metric_name']=='cash_and_securities') & (metrics['fiscal_period'].str.match(r'^FY\d{4}$'))]
_debt = metrics[(metrics['metric_name']=='total_debt') & (metrics['fiscal_period'].str.match(r'^FY\d{4}$'))]
if not _cash.empty and not _debt.empty:
    cash_val = float(_cash.sort_values('fiscal_period').iloc[-1]['metric_value'])
    debt_val = float(_debt.sort_values('fiscal_period').iloc[-1]['metric_value'])
    net_cash = cash_val - debt_val
else:
    net_cash = 0  # fallback if data unavailable
    cash_val = 0.0
    debt_val = 0.0
    print('WARNING: Could not extract net cash from XBRL data; using $0 fallback')

# Extract diluted shares from validated XBRL data
_shares = metrics[(metrics['metric_name']=='diluted_shares') & (metrics['fiscal_period'].str.match(r'^FY\d{4}$'))]
if not _shares.empty:
    shares = float(_shares.sort_values('fiscal_period').iloc[-1]['metric_value'])
else:
    shares = config.fallback_diluted_shares if hasattr(config, 'fallback_diluted_shares') and config.fallback_diluted_shares else 24_500_000_000
    print(f'WARNING: Could not extract diluted shares from XBRL data; using fallback {shares:,.0f}')

print(f'Base revenue: ${base_revenue/1e9:,.1f}B')
print(f'Net cash:     ${net_cash/1e9:,.1f}B (cash ${cash_val/1e9:,.1f}B - debt ${debt_val/1e9:,.1f}B)')
print(f'Shares:       {shares/1e9:,.2f}B diluted')

dcf = val.compute_dcf(base_assumptions, base_revenue, wacc, net_cash, shares)

print("\nDCF Output:")
print(f"  Enterprise value:  ${dcf['enterprise_value'] / 1e9:,.1f}B")
print(f"  Net cash bridge:   ${dcf['net_cash_bridge'] / 1e9:,.1f}B")
print(f"  Equity value:      ${dcf['equity_value'] / 1e9:,.1f}B")
print(f"  Per-share value:   ${dcf['per_share_value']:,.2f}")
print(f"  Terminal value:    ${dcf['terminal_value'] / 1e9:,.1f}B")
print(f"  PV of terminal:    ${dcf['pv_terminal'] / 1e9:,.1f}B")


In [ ]:
# Show year-by-year projection
projection = pd.DataFrame({
    "Year": range(1, config.projection_years + 1),
    "Revenue ($B)": [r / 1e9 for r in dcf["projected_revenue"]],
    "FCF Margin": dcf["fcf_margin"],
    "FCF ($B)": [f / 1e9 for f in dcf["projected_fcf"]],
    "Discount Factor": dcf["discount_factors"],
    "PV of FCF ($B)": [p / 1e9 for p in dcf["pv_fcf"]],
})
projection

## 2. Scenario Analysis

`build_scenarios()` runs bear / base / bull DCFs and computes a
probability-weighted expected value.

In [ ]:
val.set_base_revenue(base_revenue)
scenarios = val.build_scenarios(base_revenue, wacc, net_cash, shares)

print(f"{'Scenario':8s} {'Prob':>6s} {'CAGR':>6s} {'Margin':>8s} {'Per Share':>12s}")
print("-" * 44)
expected = 0.0
for name in ["bear", "base", "bull"]:
    s = scenarios[name]
    a = s["assumptions"]
    psv = s["per_share_value"]
    expected += psv * s["probability"]
    print(f"  {name:6s} {a.probability:>5.0%} {a.revenue_cagr:>5.0%} "
          f"{a.fcf_margin_terminal:>7.0%} ${psv:>10,.2f}")
print(f"\n  Probability-weighted value: ${expected:,.2f}")

## 3. Sensitivity Table

A 5×5 grid of per-share values across WACC and terminal growth rates.

In [ ]:
wacc_range = [0.08, 0.09, 0.10, 0.11, 0.12]
tg_range = [0.020, 0.025, 0.030, 0.035, 0.040]

sensitivity = val.compute_sensitivity_table(
    base_assumptions, base_revenue, net_cash, shares, wacc_range, tg_range
)
print("Sensitivity table (WACC × terminal growth → per-share value):")
sensitivity.round(2)

## 4. Reverse-DCF Grid

The reverse-DCF asks: *what revenue CAGR and terminal FCF margin
does the current price imply?* Cells near the current price are
highlighted — these show the market's implicit assumptions.

In [ ]:
# Load actual current price from market data
_prices = pd.read_csv('data/raw/market_prices.csv')
_nvda = _prices[_prices['ticker']=='NVDA'].sort_values('date')
current_price = float(_nvda.iloc[-1]['adj_close']) if not _nvda.empty else 208.27

cagr_range = [0.10, 0.15, 0.20, 0.25, 0.30]
margin_range = [0.20, 0.25, 0.30, 0.35, 0.40]

reverse_grid = val.compute_reverse_dcf_grid(
    price=current_price,
    shares=shares,
    cash=net_cash,
    wacc=wacc,
    cagr_range=cagr_range,
    margin_range=margin_range,
)
print("Reverse-DCF grid (CAGR × margin → implied per-share value):")
reverse_grid.round(2)


## 5. Implied Solvers

One-variable solvers find the CAGR (holding margin fixed) or margin
(holding CAGR fixed) that equates DCF value to the current price.
These are supplemental — they don't claim a single "correct" pair.

In [ ]:
implied_cagr = val.solve_implied_cagr(
    price=current_price, shares=shares, cash=net_cash,
    wacc=wacc, fixed_margin=0.30,
)
implied_margin = val.solve_implied_margin(
    price=current_price, shares=shares, cash=net_cash,
    wacc=wacc, fixed_cagr=0.20,
)

print(f"Implied CAGR (at 30% terminal margin): {implied_cagr:.1%}")
print(f"Implied margin (at 20% CAGR):          {implied_margin:.1%}")

## 6. Recommendation Eligibility and Rating

`generate_recommendation()` first checks the `RecommendationStatus` from the
eligibility gate. If the status is not `formal_rating`, the rating is automatically
set to "Not Rated" with an explanation of which gates failed.

When eligibility passes, the scorecard applies thresholds:
- **Buy**: prob-weighted upside >15%, base upside >15%, bear downside ≥ -25%
- **Sell**: downside >10%
- **Hold**: everything else

Each scorecard component shows a `ComponentStatus` (usable / diagnostic_only /
unavailable / blocked). Components with `diagnostic_only` status are excluded
from score direction but still displayed for transparency.

Includes "what must be true" statements for each rating.

In [ ]:
rec = val.generate_recommendation(
    scenarios=scenarios,
    current_price=current_price,
    reverse_dcf=reverse_grid,
    ml_signal=0.05,
    narrative_signal=0.02,
)

print(f"Rating:          {rec.rating}")
print(f"Current price:   ${rec.current_price:,.2f}")
print(f"Target price:    ${rec.target_price:,.2f}")
print(f"Upside:          {rec.upside_pct:.1%}")
print(f"Expected value:  ${rec.expected_value:,.2f}")
print(f"\nScorecard:")
for k, v in rec.scorecard.items():
    print(f"  {k:30s} {v}")
print(f"\nWhat must be true (Buy):")
print(f"  {rec.what_must_be_true_buy}")

## 7. Charts

The `ChartGenerator` produces 6–8 exhibits saved to `outputs/figures/`.
Each chart includes a source caption and generates an `ExhibitRecord`
for attribution tracking.

In [ ]:
charts = ChartGenerator(config)

# Generate scenario chart from our illustrative data
scenario_data = {
    name: {
        "per_share_value": s["per_share_value"],
        "probability": s["probability"],
    }
    for name, s in scenarios.items()
}
charts.plot_dcf_scenarios(scenario_data)
print(f"Charts generated: {len(charts.exhibits)}")
for ex in charts.exhibits:
    print(f"  {ex.exhibit_id}: {ex.title} → {ex.file_path}")

## 8. Report Generation

The `ReportGenerator` assembles all data into a Jinja2-rendered
Markdown report, then converts to PDF (or HTML fallback).

The report operates in one of three **report modes** based on the
recommendation eligibility gate:
- `formal_rating` — full report with Buy/Hold/Sell rating
- `diagnostic_not_rated` — cover page shows "Not Rated", valuation labeled diagnostic
- `failed` — pipeline error, minimal output

The critical step is `build_report_context()` — it enforces
`source_available_date <= report_date` on all data entering the report
and includes the `RecommendationStatus` to determine the report mode.

In [ ]:
report_gen = ReportGenerator(config)

# Show the point-in-time filtering mechanism
print("Report date:", config.report_date)
print("\nPoint-in-time audit examples:")
for row in report_gen._build_pit_audit_table(config.report_date):
    print(f"  {row['allowed']} {row['scenario']}")
    print(f"       {row['reason']}")

## 9. Audit and Attribution

The `AuditModule` generates:
- `source_attribution.md` — every data source traced
- `data_dictionary.md` — every variable documented
- `limitations.md` — gaps, assumptions, blockers (MUST include validation failures when present)
- Self-audit appended to `model_audit.md` — skeptical grading
- `audit_status.json` — machine-readable gate results and final package status

The **audit consistency check** (`check_audit_consistency()`) runs after report
generation and cross-validates all output files for contradictions. If any
contradiction is found, the final package status in `audit_status.json` is `failed`.

In [ ]:
audit = AuditModule(config)

print("Audit outputs:")
print(f"  outputs/source_attribution.md")
print(f"  outputs/data_dictionary.md")
print(f"  outputs/limitations.md")
print(f"  outputs/model_audit.md (with self-audit)")
print(f"\nThese are generated by the full pipeline run.")
print(f"See: python scripts/run_pipeline.py --ticker NVDA --report-date {config.report_date}")

---

## Full Pipeline

To run the complete pipeline end-to-end:

```bash
python scripts/run_pipeline.py --ticker NVDA --report-date 2026-05-01 --price-date 2026-05-01
```

This produces all outputs in `data/processed/` and `outputs/`.
These notebooks serve as explanatory walkthroughs — the pipeline
script is the reproducibility path.